In [ ]:
import pyspark
from pyspark.sql import SparkSession

In [ ]:
import ConnectionConfig as cc
cc.setupEnvironment()

### Connection properties
ConnectionConfig.py (cc) simplifies the database connection process.
Consult the file to get more insights.

# Creating the operational database
In order to run this demo you have to create a tutorial_op database and run the PostgreSQL_SalesOperational.sql script.

### Database connection
1. Make sure the postgres container is running.
2. Create a database connection in PyCharm with the following properties:
   - Host: localhost
   - Port: 5432
   - User: postgres
   - Password: data
3. Right click the connection > new > database > enter database name tutorial_op > ok
4. Refresh "data sources"
5. Make the new database visible in the database explorer: CLick on the number next to de data source > check tutorial_op > ok
6. Right click PostgreSQL_SalesOperational.sql > Run... > select the connection you created > Run
7. Check if the tables are created in the database

#### Problems
If you get "password authentication" error, you probably also run a local postgres database. See README.MD 'setup the data architecture to solve this'

### Session setup

In [ ]:
spark = cc.startLocalCluster("DeltaTableEx")
spark.getActiveSession()

In [ ]:
from delta import configure_spark_with_delta_pip

builder = SparkSession.builder \
    .appName("DBConnectionTest") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .master("local[4]")
#This one must be added to be able to query a jdbc source
#extra_packages = ["org.apache.spark:spark-sql-kafka-0-10_2.12:3.1.2","com.microsoft.sqlserver:mssql-jdbc:12.2.0.jre8"]
extra_packages = ["org.apache.spark:spark-sql-kafka-0-10_2.12:3.1.2","org.postgresql:postgresql:42.7.7"]

builder = configure_spark_with_delta_pip(builder, extra_packages=extra_packages)

spark = builder.getOrCreate()
builder.getOrCreate()
spark.sparkContext.setLogLevel("DEBUG")

In [ ]:
spark.getActiveSession()


### Reading a JDBC table
Read a table from the database connection

#### Using the ConnectionConfig (cc) to make things easy
cc gives you the possibility to store connection properties in environment variables. This way you don't have to hardcode connection properties in your code.
First set the connection profile to tutorial_op. This will set the properties url, username and password to the values that are stored in the environment variables.

#### Partitioning
As Spark is build to work in parallel reading from the database can also be done in parallel. In this case we define 4 partitions. Spark has to know how to split the data for every partition. Therefore you have to provide a partition column and a lower and upperbound. In this case the  on of the 4 queries that Spark will fire looks like "select * from dbo.sales where Order_ID <= 500 and Order_id > 250"

In [ ]:
spark.sparkContext.setLogLevel("DEBUG")

cc.set_connectionProfile("tutorial_op")
print(cc.create_jdbc())
sales_df = spark.read \
    .format("jdbc") \
    .option("driver" , "org.postgresql.Driver") \
    .option("url", cc.get_Property("url")) \
    .option("dbtable", "sales") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .option("partitionColumn", "Order_ID") \
    .option("numPartitions", 4) \
    .option("lowerBound", 0) \
    .option("upperBound", 1001) \
    .load()
sales_df.show(1000)

In [ ]:
spark.stop()